In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Set up plotting parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
plt.style.use('default')

In [2]:
# Load and Explore the Dataset
ds = xr.open_dataset('../data/2025/processed/lvl0/lvl0_main.nc')

# Explore the structure
print("Dataset overview:")
print(ds)
print("\nData variables:")
print(ds.data_vars)
print("\nCoordinates:")
print(ds.coords)

# Check data size
total_points = ds.temp_c.size
print(f"\nTotal data points: {total_points:,}")
print(f"Dataset memory usage: ~{total_points * 8 / 1024**2:.1f} MB")

Dataset overview:
<xarray.Dataset> Size: 273MB
Dimensions:            (sensor_idx: 59, datetime: 287248)
Coordinates:
  * datetime           (datetime) datetime64[ns] 2MB 2025-06-11T15:00:13 ... ...
    sensor_id          (sensor_idx) <U8 2kB ...
    site_id            (sensor_idx) <U9 2kB ...
    height             (sensor_idx) <U2 472B ...
    shielding          (sensor_idx) <U10 2kB ...
    sensor_type        (sensor_idx) <U12 3kB ...
    sensor_generation  (sensor_idx) <U3 708B ...
    elevation          (sensor_idx) float64 472B ...
    latitude           (sensor_idx) float64 472B ...
    longitude          (sensor_idx) float64 472B ...
  * sensor_idx         (sensor_idx) int64 472B 0 1 2 3 4 5 ... 53 54 55 56 57 58
Data variables:
    temp_c             (sensor_idx, datetime) float64 136MB ...
    intensity_lux      (sensor_idx, datetime) float64 136MB ...
Attributes: (12/19)
    sensor_type:                  hobo pendant
    sensor_generation:            new
    sensor_id:      

In [3]:
# Get all unique combinations that exist
existing_combos = pd.DataFrame({
    'site_id': ds['site_id'].values,
    'height': ds['height'].values,
    'shielding': ds['shielding'].values
}).drop_duplicates()

# Calculate possible combinations
n_sites = len(ds['site_id'].values)  # This is per sensor, but we want unique
n_heights = len(pd.unique(ds['height'].values))
n_shieldings = len(pd.unique(ds['shielding'].values))
n_unique_sites = len(pd.unique(ds['site_id'].values))

total_possible = n_unique_sites * n_heights * n_shieldings
actual_existing = len(existing_combos)

print(f"Unique sites: {n_unique_sites}")
print(f"Unique heights: {n_heights} {pd.unique(ds['height'].values)}")
print(f"Unique shieldings: {n_shieldings} {pd.unique(ds['shielding'].values)}")
print(f"\nTotal possible combinations: {total_possible}")
print(f"Actually existing combinations: {actual_existing}")
print(f"Sparsity: {actual_existing/total_possible:.1%} filled")
print(f"\nInflation factor if restructured: {total_possible/actual_existing:.2f}x")

# Show the existing combinations
print(f"\nExisting combinations ({len(existing_combos)}):")
print(existing_combos.sort_values(['site_id', 'height', 'shielding']))

# Check which combinations are missing
print("\n--- Missing combinations check ---")
for height in pd.unique(ds['height'].values):
    for shielding in pd.unique(ds['shielding'].values):
        count = ((ds['height'] == height) & (ds['shielding'] == shielding)).sum().values
        print(f"{height:>3} + {shielding:<12}: {count} sensors")

Unique sites: 31
Unique heights: 4 ['2m' '1m' '0m' '0.']
Unique shieldings: 3 ['shielded' 'unshielded' 'unshield']

Total possible combinations: 372
Actually existing combinations: 59
Sparsity: 15.9% filled

Inflation factor if restructured: 6.31x

Existing combinations (59):
      site_id height   shielding
16        A01     1m    shielded
38        A02     0.    unshield
37        A02     1m    shielded
39        A02     2m    shielded
27        A03     1m    shielded
26        A03     2m    shielded
20        A04     1m    shielded
21        A04     2m    shielded
12        A05     1m    shielded
11        A05     2m    shielded
32        A06     0m  unshielded
34        A06     1m    shielded
33        A06     2m    shielded
30        A07     0m    unshield
28        A07     1m    shielded
29        A07     2m    shielded
56        A08     1m    shielded
55        A08     2m    shielded
17        A10     0m  unshielded
18        A10     1m    shielded
19        A10     2m    shield

In [4]:
# Test the real inflation
import xarray as xr

# Current structure size
current_size = ds.nbytes

# Restructure to multi-dimensional
ds_multi = ds.set_index(sensor_idx=['site_id', 'height', 'shielding']).unstack('sensor_idx')

# Check actual size
multi_size = ds_multi.nbytes

print(f"Current in-memory: {current_size / 1e6:.1f} MB")
print(f"Restructured in-memory: {multi_size / 1e6:.1f} MB")
print(f"Actual inflation: {multi_size / current_size:.2f}x")

# Save both to disk with compression to see real storage cost
ds.to_netcdf('current.nc', encoding={'temp_c': {'zlib': True, 'complevel': 5},
                                      'intensity_lux': {'zlib': True, 'complevel': 5}})
ds_multi.to_netcdf('multi.nc', encoding={'temp_c': {'zlib': True, 'complevel': 5},
                                          'intensity_lux': {'zlib': True, 'complevel': 5}})

import os
print(f"\nOn-disk current: {os.path.getsize('current.nc') / 1e6:.1f} MB")
print(f"On-disk multi: {os.path.getsize('multi.nc') / 1e6:.1f} MB")

Current in-memory: 273.5 MB
Restructured in-memory: 1712.0 MB
Actual inflation: 6.26x

On-disk current: 17.1 MB
On-disk multi: 21.5 MB


In [5]:
import time

# Test 1: Current structure (must compute mask first)
ds_current = xr.open_dataset('current.nc', chunks={'datetime': None})
start = time.time()
mask = ((ds_current['site_id'] == 'A02') & (ds_current['height'] == '2m')).compute()
result1 = ds_current.isel(sensor_idx=mask)['temp_c'].mean().compute()
time1 = time.time() - start

# Test 2: Multi-dimensional
ds_multi = xr.open_dataset('multi.nc', chunks={'datetime': None})
start = time.time()
result2 = ds_multi.sel(site_id='A02', height='2m')['temp_c'].mean().compute()
time2 = time.time() - start

print(f"Current structure: {time1:.3f}s")
print(f"Multi-dimensional: {time2:.3f}s")
print(f"Speedup: {time1/time2:.2f}x")

Current structure: 0.144s
Multi-dimensional: 0.182s
Speedup: 0.79x


In [6]:
import time

# Scenario 1: Select multiple sites (common use case you mentioned)
sites = ['A02', 'A03', 'G01', 'D02']

# Current structure
ds_current = xr.open_dataset('current.nc', chunks={'datetime': None})
start = time.time()
mask = ds_current['site_id'].isin(sites).compute()
result1 = ds_current.isel(sensor_idx=mask)['temp_c'].mean().compute()
time1 = time.time() - start

# Multi-dimensional  
ds_multi = xr.open_dataset('multi.nc', chunks={'datetime': None})
start = time.time()
result2 = ds_multi.sel(site_id=sites)['temp_c'].mean().compute()
time2 = time.time() - start

print(f"Multiple sites - Current: {time1:.3f}s, Multi: {time2:.3f}s")

# Scenario 2: Iterate through all sites individually (analysis workflow)
print("\nIterating through all sites:")

start = time.time()
for site in ds_current['site_id'].values[:5]:  # First 5 as example
    mask = (ds_current['site_id'] == site).compute()
    _ = ds_current.isel(sensor_idx=mask)['temp_c'].mean().compute()
time1 = time.time() - start

start = time.time()
unique_sites = [s for s in ds_multi['site_id'].values if s is not np.nan][:5]
for site in unique_sites:
    _ = ds_multi.sel(site_id=site)['temp_c'].mean().compute()
time2 = time.time() - start

print(f"Current: {time1:.3f}s, Multi: {time2:.3f}s")

Multiple sites - Current: 0.227s, Multi: 2.087s

Iterating through all sites:
Current: 0.606s, Multi: 3.416s


In [7]:
import time

# Rechunk: entire timeseries together, chunk by sensor combinations
ds_multi_rechunked = ds_multi.chunk({'datetime': -1,      # Full timeseries together
                                      'site_id': 1,        # One site per chunk
                                      'height': 1,         # One height per chunk  
                                      'shielding': 1})     # One shielding per chunk

ds_multi_rechunked.to_netcdf('multi_timeseries_chunks.nc', 
                              encoding={'temp_c': {'zlib': True, 'complevel': 5},
                                       'intensity_lux': {'zlib': True, 'complevel': 5}})

print(f"File size: {os.path.getsize('multi_timeseries_chunks.nc') / 1e6:.1f} MB")

# Now retest
print("\n=== Performance Test with Timeseries Chunking ===\n")

ds_current = xr.open_dataset('current.nc', chunks={'datetime': -1})
ds_multi = xr.open_dataset('multi_timeseries_chunks.nc')

# Test 1: Single site
start = time.time()
mask = ((ds_current['site_id'] == 'A02') & (ds_current['height'] == '2m')).compute()
result1 = ds_current.isel(sensor_idx=mask)['temp_c'].mean().compute()
time1 = time.time() - start

start = time.time()
result2 = ds_multi.sel(site_id='A02', height='2m')['temp_c'].mean().compute()
time2 = time.time() - start

print(f"Single site - Current: {time1:.3f}s, Multi: {time2:.3f}s")

# Test 2: Multiple sites (your common pattern)
sites = ['A02', 'A03', 'G01', 'D02']

start = time.time()
mask = ds_current['site_id'].isin(sites).compute()
result1 = ds_current.isel(sensor_idx=mask)['temp_c'].mean().compute()
time1 = time.time() - start

start = time.time()
result2 = ds_multi.sel(site_id=sites)['temp_c'].mean().compute()
time2 = time.time() - start

print(f"Multiple sites - Current: {time1:.3f}s, Multi: {time2:.3f}s")

# Test 3: What about just loading a subset to analyze?
start = time.time()
subset = ds_multi.sel(site_id=['A02', 'A03'])
_ = subset['temp_c'].values  # Force load
time_multi = time.time() - start

start = time.time()
mask = ds_current['site_id'].isin(['A02', 'A03']).compute()
subset = ds_current.isel(sensor_idx=mask)
_ = subset['temp_c'].values
time_current = time.time() - start

print(f"Load subset to memory - Current: {time_current:.3f}s, Multi: {time_multi:.3f}s")

File size: 21.5 MB

=== Performance Test with Timeseries Chunking ===

Single site - Current: 0.127s, Multi: 0.175s
Multiple sites - Current: 0.261s, Multi: 2.830s
Load subset to memory - Current: 0.119s, Multi: 0.722s
